In [21]:
import torchaudio
torchaudio.set_audio_backend("soundfile")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [45]:
import torchaudio
torchaudio.set_audio_backend("soundfile")

C:\Users\Andrei\AppData\Local\Temp\ipykernel_1380\2654335069.py:2: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")


In [46]:
import torch
import torchaudio
from tqdm import tqdm
from sklearn.metrics import f1_score

# -------- CONFIG --------
SAMPLE_RATE = 16000

# MFCC (lightweight)
mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=SAMPLE_RATE,
    n_mfcc=40,  # standard choice
    melkwargs={
        "n_fft": 400,
        "hop_length": 160,
        "n_mels": 64
    }
)

# Log-Mel Spectrogram (heavy model)
mel_spec_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=400,
    hop_length=160,
    n_mels=128
)

amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

In [47]:
def extract_features(waveform, mode="mfcc"):
    """
    waveform: Tensor [1, T]
    mode: 'mfcc' or 'mel'
    """
    
    if mode == "mfcc":
        features = mfcc_transform(waveform)  # [1, n_mfcc, time]
        
    elif mode == "mel":
        features = mel_spec_transform(waveform)  # [1, n_mels, time]
        features = amplitude_to_db(features)
        
    else:
        raise ValueError("Unknown mode")

    return features.squeeze(0)  # [feat_dim, time]

In [48]:
MAX_LEN = SAMPLE_RATE * 6  # 6 seconds


def fix_length(waveform):
    """
    waveform: [1, T]
    """
    T = waveform.shape[1]

    if T > MAX_LEN:
        # truncate
        waveform = waveform[:, :MAX_LEN]
    else:
        # pad
        pad_size = MAX_LEN - T
        waveform = torch.nn.functional.pad(waveform, (0, pad_size))

    return waveform

In [49]:
import librosa
import torch

def load_audio(path):
    # librosa loads reliably on Windows
    y, sr = librosa.load(path, sr=SAMPLE_RATE)

    # convert to torch
    waveform = torch.tensor(y, dtype=torch.float32).unsqueeze(0)

    waveform = fix_length(waveform)

    return waveform

In [50]:
import pandas as pd
from torch.utils.data import Dataset
from pathlib import Path

# -------- PATHS --------
DATA_ROOT = Path("../Data/MELD.Audio")

# label mapping (important)
EMO_MAP = {
    "joy": "happiness"
}

# final label set (fixed order!)
LABELS = [
    "anger",
    "disgust",
    "fear",
    "happiness",
    "sadness",
    "surprise",
    "neutral"
]

label2idx = {l: i for i, l in enumerate(LABELS)}

In [118]:
class MELDAudioDataset(Dataset):
    def __init__(self, split="train", mode="mfcc"):
        self.split = split
        self.mode = mode

        # load CSV
        if split == "train":
            csv_path = "../Data/MELD.Raw/train/train_sent_emo.csv"
        elif split == "dev":
            csv_path = "../Data/MELD.Raw/dev_sent_emo.csv"
        else:
            csv_path = "../Data/MELD.Raw/test_sent_emo.csv"

        df = pd.read_csv(csv_path)

        # normalize labels
        df["Emotion"] = df["Emotion"].str.lower()
        df["Emotion"] = df["Emotion"].replace(EMO_MAP)

        self.audio_dir = DATA_ROOT / split

        # filter missing files
        valid_rows = []
        for _, row in df.iterrows():
            dia = row["Dialogue_ID"]
            utt = row["Utterance_ID"]

            filename = f"dia{dia}_utt{utt}.wav"
            path = self.audio_dir / filename

            if path.exists():
                valid_rows.append(row)

        self.df = pd.DataFrame(valid_rows).reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        dia = row["Dialogue_ID"]
        utt = row["Utterance_ID"]

        filename = f"dia{dia}_utt{utt}.wav"
        path = self.audio_dir / filename

        waveform = load_audio(path)
        features = extract_features(waveform, self.mode)

        label = label2idx[row["Emotion"]]

        return features, label

In [101]:
train_ds = MELDAudioDataset("train")
print(len(train_ds))
x, y = train_ds[0]

9988


e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [102]:
torchaudio.get_audio_backend()

C:\Users\Andrei\AppData\Local\Temp\ipykernel_1380\1746710290.py:1: UserWarning: torchaudio._backend.get_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.get_audio_backend()


In [103]:
ds = MELDAudioDataset(split="train", mode="mfcc")

x, y = ds[0]

print(x.shape)  # [feat_dim, time]
print(y)        # int

torch.Size([40, 601])
6


Small CNN on MFCC (fast, low params, decent performance)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class AudioCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.dropout = nn.Dropout(0.3)

        self.pool = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(24000, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # x: [B, feat, time] → add channel
        x = x.unsqueeze(1)  # [B, 1, F, T]

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [105]:
model = AudioCNN()

x = torch.randn(1, 40, 601)
out = model.conv1(x.unsqueeze(1))
out = model.pool(out)
out = model.pool(model.conv2(out))
out = model.pool(model.conv3(out))

print(out.shape)

torch.Size([1, 64, 5, 75])


In [106]:
model = AudioCNN()

x = torch.randn(2, 40, 601)  # batch of 2
out = model(x)

print(out.shape)  # should be [2, 7]

torch.Size([2, 7])


## Data loader

In [107]:
from torch.utils.data import DataLoader

train_ds = MELDAudioDataset(split="train", mode="mfcc")
dev_ds   = MELDAudioDataset(split="dev", mode="mfcc")

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
dev_loader   = DataLoader(dev_ds, batch_size=32, shuffle=False)

In [108]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [109]:
model = AudioCNN().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2+cu118
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Training loop

In [110]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        out = model(x)
        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

Evaluation

In [111]:
def evaluate(model, loader):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            out = model(x)
            preds = out.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    return f1

Train

In [40]:
EPOCHS = 10

best_f1 = 0

for epoch in tqdm(range(EPOCHS)):
    train_loss = train_one_epoch(model, train_loader)
    val_f1 = evaluate(model, dev_loader)

    print(f"Epoch {epoch+1}: loss={train_loss:.4f}, val_f1={val_f1:.4f}")

    # save best model
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "../Models/Audio_Models/best_audio_model.pth")

 10%|█         | 1/10 [00:20<03:05, 20.64s/it]

Epoch 1: loss=1.9228, val_f1=0.2518


 20%|██        | 2/10 [00:42<02:50, 21.33s/it]

Epoch 2: loss=1.5349, val_f1=0.2518


 30%|███       | 3/10 [01:04<02:31, 21.61s/it]

...

Epoch 8: loss=1.2395, val_f1=0.2857


 90%|█████████ | 9/10 [03:15<00:21, 21.79s/it]

Epoch 9: loss=1.0981, val_f1=0.3066


100%|██████████| 10/10 [03:36<00:00, 21.65s/it]

Epoch 10: loss=0.9394, val_f1=0.2964


In [112]:
train_ds = MELDAudioDataset(split="train", mode="mel")
dev_ds   = MELDAudioDataset(split="dev", mode="mel")

In [135]:
class AudioCNNHeavy(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)

        self.pool = nn.MaxPool2d(2)

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.dropout = nn.Dropout(0.4)

        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))

        # ✅ reduce to fixed size
        x = self.global_pool(x)      # [B, 256, 1, 1]
        x = x.view(x.size(0), -1)    # [B, 256]

        x = self.dropout(x)
        x = self.fc(x)

        return x

In [136]:
model = AudioCNNHeavy()

x = torch.randn(1, 128, 601)
out = model.conv1(x.unsqueeze(1))
out = model.pool(out)
out = model.pool(model.conv2(out))
out = model.pool(model.conv3(out))
out = model.pool(model.conv4(out))

print(out.shape)

torch.Size([1, 256, 8, 37])


In [137]:
model = AudioCNNHeavy()
x = torch.randn(2, 128, 601)

out = model(x)
print(out.shape)  # should be [2, 7]

torch.Size([2, 7])


In [138]:
model = AudioCNNHeavy().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)  # lower LR
criterion = nn.CrossEntropyLoss()

In [153]:
%pip uninstall numpy -y
%pip install numpy==1.26.4

Found existing installation: numpy 2.3.5
Uninstalling numpy-2.3.5:
  Successfully uninstalled numpy-2.3.5
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.


  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
torchvision 0.22.1+cu118 requires torch==2.7.1+cu118, but you have torch 2.2.2+cu118 which is incompatible.


In [42]:
best_acc = 0

for epoch in range(100):
    model.train()
    total_loss = 0.0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        out = model(x)
        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    # evaluation
    model.eval()
    correct = 0
    total = 0

    if val_macro_f1 > best_f1:
        best_f1 = val_macro_f1
        torch.save(model.state_dict(), "best_audio_weighted.pth")
        print("Saved best model.")

            out = model(x)
            preds = out.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    val_acc = correct / total

    print(f"Epoch {epoch+1}: loss={train_loss:.4f}, val_acc={val_acc:.4f}")

    # save best
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "../Models/Audio_Models/best_audio_heavy.pth")

  0%|          | 0/10 [00:00<?, ?it/s]